# 03 — Similarity Search with Embeddings

**Goal**: Build a semantic search engine from scratch.

By the end of this notebook, you'll have a working "search by meaning" system.
This is the foundation of **Retrieval Augmented Generation (RAG)** — the most important GenAI pattern.

In [ ]:
import numpy as np  # Numerical computing: arrays, matrix ops
import pandas as pd  # Data handling: organize search results in tables
from sentence_transformers import SentenceTransformer  # Generate embeddings for documents & queries
from sklearn.metrics.pairwise import cosine_similarity  # Rank results by semantic similarity

model = SentenceTransformer('all-MiniLM-L6-v2')
print(f"Model ready. Dim: {model.get_sentence_embedding_dimension()}")

## 1. Our Document Collection

Let's create a small set of documents about various topics.

In [ ]:
documents = [
    "The transformer architecture uses self-attention to process sequences",
    "Transfer learning allows fine-tuning pretrained models on new tasks",
    "Retrieval Augmented Generation combines search with LLM generation",
    "Vector databases store embeddings for fast similarity search",
    "LoRA is a parameter-efficient fine-tuning technique that adds adapters",
    "LangChain provides tools for building LLM-powered applications",
    "LlamaIndex specializes in data indexing and retrieval for LLMs",
    "ChromaDB is an open-source vector database written in Python",
    "Prompt engineering is the practice of designing effective LLM prompts",
    "Ollama runs large language models locally on your machine",
    "Gradient descent optimizes neural network parameters",
    "Backpropagation computes gradients through the network layers",
    "Convolutional neural networks excel at image recognition tasks",
    "Recurrent neural networks process sequential data with hidden states",
    "The attention mechanism allows models to focus on relevant parts"
]

print(f"{len(documents)} documents loaded")

## 2. Embed All Documents

We compute embeddings for every document **once** (this is indexing).

In [ ]:
doc_embeddings = model.encode(documents)
print(f"Document embeddings shape: {doc_embeddings.shape}")

## 3. Search Function

When a user asks a question:
1. Embed the query
2. Compute similarity with all document embeddings
3. Return the top-k most similar documents

In [ ]:
def search(query, top_k=3):
    """Search documents by semantic similarity."""
    query_emb = model.encode([query])
    similarities = cosine_similarity(query_emb, doc_embeddings)[0]
    
    top_indices = np.argsort(similarities)[::-1][:top_k]
    
    print(f"Query: {query}\n")
    for i, idx in enumerate(top_indices):
        print(f"{i+1}. [{similarities[idx]:.3f}] {documents[idx]}")
    return top_indices

In [ ]:
search("How do I store embeddings for my AI application?")

In [ ]:
search("What technique makes fine-tuning large models efficient?")

In [ ]:
search("Tell me about transformer models and attention")

## 4. Understanding Distance Metrics

Three common ways to measure vector similarity:

In [ ]:
from numpy.linalg import norm  # Compute vector magnitude for similarity metrics

def cosine_sim(a, b):
    return np.dot(a, b) / (norm(a) * norm(b))

def euclidean_dist(a, b):
    return np.sqrt(np.sum((a - b) ** 2))

def dot_product(a, b):
    return np.dot(a, b)

# Compare two embeddings
v1 = doc_embeddings[0]  # transformer doc
v2 = doc_embeddings[4]  # LoRA doc
v3 = doc_embeddings[14] # attention doc

print(f"Comparing 'transformer' with 'attention':")
print(f"  Cosine similarity:  {cosine_sim(v1, v3):.3f}")
print(f"  Euclidean distance: {euclidean_dist(v1, v3):.3f}")
print(f"  Dot product:        {dot_product(v1, v3):.3f}")

print(f"\nComparing 'transformer' with 'LoRA' (unrelated):")
print(f"  Cosine similarity:  {cosine_sim(v1, v2):.3f}")
print(f"  Euclidean distance: {euclidean_dist(v1, v2):.3f}")
print(f"  Dot product:        {dot_product(v1, v2):.3f}")

## 5. Adding Metadata Filtering

Real systems combine semantic search with metadata filters.

In [ ]:
# Create a richer dataset with metadata
docs_with_meta = [
    {"text": "The transformer architecture uses self-attention", "category": "architecture", "difficulty": "advanced"},
    {"text": "LoRA is a parameter-efficient fine-tuning technique", "category": "fine-tuning", "difficulty": "intermediate"},
    {"text": "Vector databases store embeddings for similarity search", "category": "infrastructure", "difficulty": "beginner"},
    {"text": "Prompt engineering designs effective LLM prompts", "category": "practical", "difficulty": "beginner"},
    {"text": "Backpropagation computes gradients through layers", "category": "theory", "difficulty": "advanced"},
]

texts = [d["text"] for d in docs_with_meta]
embeddings = model.encode(texts)

def search_with_filter(query, category=None, top_k=3):
    query_emb = model.encode([query])[0]
    
    # Filter by category if specified
    candidates = [(i, d) for i, d in enumerate(docs_with_meta)
                  if category is None or d["category"] == category]
    
    results = []
    for idx, doc in candidates:
            sim = cosine_sim(query_emb, embeddings[idx])
            results.append((sim, doc))
    
    results.sort(reverse=True)
    
    print(f"Query: {query}")
    if category:
        print(f"Filter: category={category}")
    print()
    for sim, doc in results[:top_k]:
        print(f"  [{sim:.3f}] {doc['text']} ({doc['category']})")

search_with_filter("How do I train a model?", category="beginner")

## 6. Performance Considerations

For our 15 documents, brute force (comparing with every doc) is fine.
For **millions** of documents, we need Approximate Nearest Neighbor (ANN) search.

We'll cover this in Module 4 (Vector Databases).

## Key Takeaways

1. **Semantic search = Embed query + Find nearest vectors**
2. **Cosine similarity** is the default metric for embeddings
3. **Metadata filtering** improves relevance
4. This exact pattern powers **RAG** — you just add an LLM to generate answers from retrieved docs
5. For scale, you need a vector database (chroma, pinecone, weaviate)

**Next**: Transformers & LLMs — what happens *inside* the model you've been prompting →